# Find bounding box for data

#### Project bounding box for all data:

| boundary | value | 
|----------|-------|
| West longitude | -85.94712712079293 |
| East longitide | -85.3443621648922 |
| South_latitude | 37.99712528351634 |
| North_latitude | 38.38023822809115 |

#### Span

| | difference | approx span | 
|-|------------|-------------|
| ∆ longitude | 0.6027649559007244 | 30 miles |
| ∆ latitude | 0.3831129445748118 | 26 miles |

\* Code for finding the boundaries begins below.

In [33]:

import json

from os import path

import pandas as pd
import numpy as np
from numpy import radians, sin, cos, arcsin, sqrt


#HOME = "~/code/county_coverage/"

from pyproj import CRS
from pyproj.transformer import Transformer 

class converter:
    def __init__(self):
        self.KY_grid_CRS = KYCRS = CRS("ESRI:102679")
        self.long_lat_CRS = LLCRS =CRS("epsg:4326")

        to_grid = Transformer.from_crs(crs_from=LLCRS, crs_to=KYCRS, always_xy=True)
        to_ll = Transformer.from_crs(crs_from=KYCRS, crs_to=LLCRS, always_xy=True)

        self.to_grid = self.from_ll = to_grid.transform
        self.to_ll = self.from_grid = to_ll.transform

convert_points = converter()

def central_angle(point1, point2):
    """
    Calculate the central angle between two points on the earth 
    (specified in decimal degrees)
    
    """
    lon1, lat1 = map(radians, point1)
    lon2, lat2 = map(radians, point2)
    
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    
    havTheta = sin(dlat/2.0)**2 + cos(lat1) * cos(lat2) * sin(dlon/2.0)**2
    return 2 * arcsin(sqrt(havTheta))

earth_radius_mi = 3956


In [34]:
# general functions

def get_data(filepath) -> dict:
    fp = path.join(filepath)
    with open(fp, 'r') as data:
        json_data = json.load(data)
    return json_data


def sort_long_lat(points:list) -> pd.Series:
    """Create bounding box from a list of (longitude, latitude) points."""
    longitudes = set()
    latitudes = set()
    for long, lat in points:
        longitudes.add(long)
        latitudes.add(lat)

    return pd.Series({"west_longitude": min(longitudes), "east_longitude": max(longitudes),
            "south_latitude": min(latitudes), "north_latitude": max(latitudes)})

# data is in GeoJSON data contain an array of features, each of which has properties and geometry
# We are only concerned about the geometry, which has two properties, `type` and `coordinates`. `type` is not particularly helpful here. 


def get_geometries(json_data):
    """Pull all geometry objects out of GeoJSON, ignoring other data."""
    for feature in json_data['features']:
        yield feature['geometry']['coordinates']


Source data for county boundary is a GeoJSON file that looks like this:

```json
{
"type": "FeatureCollection",
"name": "Louisville_Metro_KY_County_Boundaries",
"crs": { "type": "name", "properties": { "name": "urn:ogc:def:crs:OGC:1.3:CRS84" } },
"features": [...]}
```

Where each item in `"features"` represent a county in the Louisville, KY metro area. This includes Jefferson County, where Louisville is, and several surrounding counties. Each `feature` or county object looks like this:

```json
{ "type": "Feature", 
  "properties": { "OBJECTID": 7, 
                  "CNTY_NAME": "JEFFERSON", 
                  "FIPS": "21111", 
                  "STATE_FIPS": "21", 
                  "CNTY_FIPS": "111", 
                  "SHAPEAREA": 11083783720.6446, 
                  "SHAPELEN": 513054.30366378697 }, 
  "geometry": { "type": "Polygon", 
                "coordinates": [ [ [ -85.575811868593064, 38.334545868470848 ], 
                                   [ -85.578070603648868, 38.335714204871564 ], 
                                   [ -85.579003771392806, 38.336193264774884 ],
                                    ... ] ] } }
```

Jefferson County is the one we are interested in here. The value for `"coordinates"` is a list of lists. Each interior list contains points defining a polygon that represents the boundary of the county. The points are listed as `[longitude, latitude]` pairs. The `coordinates` are really all we need, once we find the correct county object in the list of `features`.

In [35]:
METRO_BOUNDARIES = "data/raw/Louisville_Metro_KY_County_Boundaries.geojson"

def get_JEFFCO_data(path_to_data):
    data = get_data(path_to_data)
    counties = data['features']
    # Each feature represents a county boundary. Need to find the right one.
    # The county I am interested in is called Jefferson.

    for county in counties:
        if county['properties']['CNTY_NAME'] == 'JEFFERSON':
            JEFFCO = county
            break
    return JEFFCO

JEFFCO = get_JEFFCO_data(METRO_BOUNDARIES)

JEFFCO_boundary = JEFFCO['geometry']['coordinates'][0] 
# [0] is necessary b/c geometry type is polygon, which can have multiple discontinuous
# shapes, each represented by a separate line.
# Jefferson county has no such complication: it's just one shape 

boxes = pd.DataFrame(sort_long_lat(JEFFCO_boundary), 
                     columns=pd.MultiIndex.from_tuples([('county', 'meta')]))
boxes

,county
,meta
west_longitude,-85.947127
east_longitude,-85.404922
south_latitude,37.997125
north_latitude,38.380238


In [ ]:
# Get bounding box for intersections

# Add bounding box from intersection metadata
"""Extents  ►
Extent 
Geographic extent 
Bounding rectangle 
Extent type  Extent used for searching
* West longitude -85.945347
* East longitude -85.344499
* North latitude 38.378034
* South latitude 38.005894
* Extent contains the resource Yes

Extent in the item's coordinate system 
* West longitude 1154395.500000
* East longitude 1325086.990000
* South latitude 188677.437500
* North latitude 321629.781250
* Extent contains the resource Yes"""

# west_longitude_co = 1154395.500000
# east_longitude_co = 1325086.990000
# south_latitude_co = 188677.437500
# north_latitude_co = 321629.781250
# #* Extent contains the resource Yes

LL = pd.Series( {"west_longitude": -85.945347, "east_longitude": -85.344499,
              "north_latitude": 38.378034, "south_latitude": 38.005894})

state_grid = pd.Series({'west_longitude': 1154395.500000, 'east_longitude': 1325086.990000,
             'south_latitude': 188677.437500,'north_latitude': 321629.781250})

boxes['int_meta', 'LL'] = LL
#boxes['int_meta', 'state_grid'] = state_grid

boxes

,county,int_meta
,meta,LL
west_longitude,-85.947127,-85.945347
east_longitude,-85.404922,-85.344499
south_latitude,37.997125,38.005894
north_latitude,38.380238,38.378034


In [37]:

def xy_points_from_box(series):
    xx, yy = list(), list()
    for x in (series.west_longitude, series.east_longitude):
        for y in (series.north_latitude, series.south_latitude):
            xx.append(x)
            yy.append(y)
    return xx, yy
            
cp = convert_points.to_ll(*xy_points_from_box(state_grid))

sort_long_lat(zip(*cp))

def convert_box(series, to='ll'):
    xx, yy = list(), list()
    for x in (series.west_longitude, series.east_longitude):
        for y in (series.north_latitude, series.south_latitude):
            xx.append(x)
            yy.append(y)
    if to == 'll':
        convert = convert_points.to_ll(xx, yy)
    elif to == 'grid':
        convert = convert_points.to_grid(xx, yy)
    return sort_long_lat(zip(*convert))

#convert_box(state_grid, to='ll')
boxes['int_derived', 'state_grid_convert'] = convert_box(state_grid, to='ll')
boxes

,county,int_meta,int_derived
,meta,LL,state_grid_convert
west_longitude,-85.947127,-85.945347,-85.945347
east_longitude,-85.404922,-85.344499,-85.344499
south_latitude,37.997125,38.005894,38.005894
north_latitude,38.380238,38.378034,38.378034


In [38]:
# Derive box from intersection data

INTERSECTIONS = "data/raw/intersections/Jefferson_County_KY_Street_Intersections.geojson"
intersection_data = get_data(INTERSECTIONS)
intersection_points = get_geometries(intersection_data)

# boundaries['intersections derived'] = sort_long_lat(intersection_points)
# #boundaries

boxes['int_derived', 'LL'] = sort_long_lat(intersection_points)
boxes

county   int_meta        int_derived           
                     meta         LL state_grid_convert         LL
west_longitude -85.947127 -85.945347         -85.945347 -85.936859
east_longitude -85.404922 -85.344499         -85.344499 -85.347451
south_latitude  37.997125  38.005894          38.005894  38.005901
north_latitude  38.380238  38.378034          38.378034  38.375609

In [39]:
def get_xy(data):
    for feature in data['features']:
        properties = feature['properties']
        x = properties['X_COORD']
        y= properties['Y_COORD']
        yield x, y

derived_grid = sort_long_lat(get_xy(intersection_data)) # identical to stated extent
state_grid - derived_grid


west_longitude     0.00000
east_longitude     0.00000
south_latitude     0.00000
north_latitude   -94.03625
dtype: float64

In [40]:

boxes['int_derived', 'derived_grid_box'] = convert_box(derived_grid,to='ll')
boxes

county   int_meta        int_derived             \
                     meta         LL state_grid_convert         LL   
west_longitude -85.947127 -85.945347         -85.945347 -85.936859   
east_longitude -85.404922 -85.344499         -85.344499 -85.347451   
south_latitude  37.997125  38.005894          38.005894  38.005901   
north_latitude  38.380238  38.378034          38.378034  38.375609   

                                 
               derived_grid_box  
west_longitude       -85.945353  
east_longitude       -85.344499  
south_latitude        38.005894  
north_latitude        38.378292

In [41]:
# Get bounding box for centerlines
CENTERLINES_PATH = "data/raw/centerlines/Jefferson_County_KY_Street_Centerlines.geojson"

centerline_geometries = get_geometries(get_data(CENTERLINES_PATH))
centerline_points = (point for geometry in centerline_geometries for point in geometry)

boxes['centerlines', 'derived'] = sort_long_lat(centerline_points)
boxes

county   int_meta        int_derived             \
                     meta         LL state_grid_convert         LL   
west_longitude -85.947127 -85.945347         -85.945347 -85.936859   
east_longitude -85.404922 -85.344499         -85.344499 -85.347451   
south_latitude  37.997125  38.005894          38.005894  38.005901   
north_latitude  38.380238  38.378034          38.378034  38.375609   

                                centerlines  
               derived_grid_box     derived  
west_longitude       -85.945353  -85.942405  
east_longitude       -85.344499  -85.344362  
south_latitude        38.005894   38.000584  
north_latitude        38.378292   38.377076

In [43]:
# compare boundary values to find the largest bounding box that covers all the data

BT = boxes.T
BT

west_longitude  east_longitude  \
county      meta                    -85.947127      -85.404922   
int_meta    LL                      -85.945347      -85.344499   
int_derived state_grid_convert      -85.945347      -85.344499   
            LL                      -85.936859      -85.347451   
            derived_grid_box        -85.945353      -85.344499   
centerlines derived                 -85.942405      -85.344362   

                                south_latitude  north_latitude  
county      meta                     37.997125       38.380238  
int_meta    LL                       38.005894       38.378034  
int_derived state_grid_convert       38.005894       38.378034  
            LL                       38.005901       38.375609  
            derived_grid_box         38.005894       38.378292  
centerlines derived                  38.000584       38.377076

In [44]:

comparisons = pd.Series({'west_longitude': BT.west_longitude.min(), 'east_longitude': BT.east_longitude.max(),
               'south_latitude': BT.south_latitude.min(), 'north_latitude': BT.north_latitude.max()},
               name = 'comparison')

pd.concat((boxes, comparisons), axis=1)

,"(county, meta)","(int_meta, LL)","(int_derived, state_grid_convert)","(int_derived, LL)","(int_derived, derived_grid_box)","(centerlines, derived)",comparison
west_longitude,-85.947127,-85.945347,-85.945347,-85.936859,-85.945353,-85.942405,-85.947127
east_longitude,-85.404922,-85.344499,-85.344499,-85.347451,-85.344499,-85.344362,-85.344362
south_latitude,37.997125,38.005894,38.005894,38.005901,38.005894,38.000584,37.997125
north_latitude,38.380238,38.378034,38.378034,38.375609,38.378292,38.377076,38.380238


#### Conclusion for bounding box Based on longitudes, latitudes

Official county boundary covers almost everything, except for `east_longitude` (maximum longitude), where some of the centerlines must have points east of the official county boundary. I also know that some of the intersections lie outside of Jefferson county.

The largest bounding box is represented by the `comparison` column in the last dataframe. This bounding box covers all the data across all the different input files. 

In [ ]:
LL_box = comparisons.copy()
LL_box.name = 'LL_box'
LL_box

def get_spans(box):
    out = {"longitude": abs(box.east_longitude - box.west_longitude),
            "latitude": abs(box.north_latitude - box.south_latitude)}
    return pd.Series(out, name='delta')

distance_function = lambda p1, p2:central_angle(p1, p2)*earth_radius_mi

north, south = LL_box.north_latitude, LL_box.south_latitude
east, west = LL_box.east_longitude, LL_box.west_longitude
d_north = distance_function((west, north), (east, north))
d_south = distance_function((west, south), (east, south))

d_lon = max(d_north, d_south) 
d_north < d_south # makes sense for nothern hemisphere.
d_lon

d_east = distance_function((east, south), (east, north))
d_west = distance_function((west, north), (west, south))

d_east == d_west # True
d_lat = d_east # doesn't matter which
d_lat


np.float64(41.666239235341344)

In [147]:

hypoteneuse_LL = distance_function((south, west), (north, east))
hypoteneuse_LL

d_sglon = (state_grid.west_longitude - state_grid.east_longitude)
d_sglat = (state_grid.south_latitude - state_grid.north_latitude)

hypoteneuse_SG = sqrt(d_sglon**2 + d_sglat**2)/5280

hypoteneuse_LL - hypoteneuse_SG


np.float64(0.688852331309505)

In [133]:
spans = pd.DataFrame(get_spans(LL_box))
spans['hav distance (mi)'] = [d_lon, d_lat]
spans['grid distance (mi)'] = get_spans(state_grid)/5280
spans

,delta,hav distance (mi),grid distance (mi)
longitude,0.602765,32.796693,32.327934
latitude,0.383113,26.452120,25.180368


In [135]:

get_spans(derived_grid)/5280


longitude    32.327934
latitude     25.198178
Name: delta, dtype: float64

In [ ]:
from common.county_geometry import *

east_point = (east_longitude, north_latitude)
west_point = (west_longitude, north_latitude)

long_mi = haversine_distance_mi(east_point, west_point)
long_km = haversine_distance_km(east_point, west_point)

display(f"longitude span = {long_mi} miles == {long_km} kilometers.")


north_point = (west_longitude, north_latitude)
south_point = (west_longitude, south_latitude)

lat_mi = haversine_distance_mi(south_point, north_point)
lat_km = haversine_distance_km(south_point, north_point)

display(f'latitude span = {lat_mi} miles == {lat_km} kilometers.')

'longitude span = 32.62464358670343 miles == 52.50786292126914 kilometers.'

'latitude span = 26.45211953861136 miles == 42.57346943941823 kilometers.'

In [136]:
# TODO convert centerlines geometry -> first, last -> state grid for distance -> ft distance



##### Info about X_COORD YCOORD system

via: https://www.lojic.org/data/projection-information

For this system, the Commonwealth shall be divided into a north zone and a south zone. The north zone shall be a Lambert conformal conic projection of the North American Datum of 1983, having standard parallels at north latitudes 37 degrees, 58 minutes, and 38 degrees, 58 minutes along which parallels the scale shall be exact. The origin of coordinates shall be at the intersection of the meridian 84 degrees, 15 minutes west of Greenwich, and the parallel 37 degrees, 30 minutes north latitude. This origin shall be given the coordinates: N=0, E=500,000.000 meters. The south zone shall be a Lambert conformal conic projection of the North American Datum of 1983, having standard parallels at north latitudes 36 degrees, 44 minutes, and 37 degrees, 56 minutes along which parallels the scale shall be exact. The origin of coordinates shall be at the intersection of the meridian 85 degrees, 45 minutes west of Greenwich, and the parallel 36 degrees, 20 minutes north latitude. This origin shall be given the coordinates: N=500,000.000, E=500,000.000 meters. The southern edge of the following counties shall delineate the boundary between the north zone and the south zone: Bullitt, Spencer, Anderson, Woodford, Jessamine, Fayette, Clark, Montgomery, Menifee, Morgan, and Lawrence.

One U. S. survey foot equals (1200)/(3937) meter. For conversion of meters to U. S. survey feet, multiply the meters by 3.28083333333 to twelve (12) significant figures. When converting from meters to feet, the conversion factor defined by the U. S. survey foot shall be used.


The plane coordinate values for a point on the earth's surface, used to express the geographic position or location of the point in the appropriate zone of this system, shall consist of two (2) distances expressed in U. S. survey feet and decimals of a foot when using the Kentucky Coordinate System of 1983. For the Kentucky Coordinate System of 1983, one (1) of the distances, to be known as the "northing" or "N", shall give the position in a north/south direction. The other, to be known as the "easting" or "E" shall give the position in an east/west direction. These coordinates shall be made to depend upon and conform to plane rectangular coordinates values for the monumented points of the North American National Geodetic Horizontal Network as published by the National Ocean Service/National Geodetic Survey, and whose plane coordinates have been computed on the systems established by the National Ocean Service/National Geodetic Survey. Any such station may be used for establishing a survey connection to the Kentucky Coordinate System of 1983.